<a href="https://colab.research.google.com/github/nsr51324/flyrank-ml-internship/blob/main/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsr51324/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: aggregate per content item inside DuckDB, then hand the small result
to pandas/sklearn.

**Modified for Task 1 + Task 2:**
- Window widened from 30/60 days to a **90-day window vs. the prior 90 days** (180 days of raw
  data touched instead of 60).
- `HAVING imp_prev90 >= 100` kept as the activity threshold — change it if you want a stricter or
  looser panel.
- New engineered features pulled straight out of SQL instead of computed later in pandas:
  - `position_change` = `pos_last90 - pos_prev90` (negative = ranking improved, i.e. the average
    position number got smaller; positive = ranking got worse).
  - `ctr_prev90` = clicks / impressions in the prior window — a page's baseline efficiency before
    the outcome window.
  - `weekend_share` = the fraction of prior-window impressions that landed on a Saturday/Sunday
    (ISO weekday 6/7) — a proxy for whether a page's traffic is B2C/leisure-shaped vs. B2B/weekday-shaped.
  - `impression_volatility` = day-to-day standard deviation of impressions in the prior window,
    normalized by the average daily impressions — a noisy page is harder to trust for momentum
    prediction than a stable one.

This is the heaviest cell in the notebook — expect a bit longer than the 30-day version since it
scans 180 days of raw data instead of 60. If it runs past ~10 minutes or errors with HTTP 429,
re-run this section against `TABLES['fact_daily_sample']` first.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last90,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev90,
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last90,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev90,
            AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 90 DAY THEN f.gsc_avg_position END)       AS pos_last90,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY THEN f.gsc_avg_position END)       AS pos_prev90,
            STDDEV(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY THEN f.gsc_impressions END)     AS imp_prev90_std,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 90 DAY
                      AND isodow(f.report_date) IN (6, 7) THEN f.gsc_impressions ELSE 0 END)             AS imp_prev90_weekend
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 180 DAY
        GROUP BY 1, 2
        HAVING imp_prev90 >= 100
    )
    SELECT
        *,
        pos_last90 - pos_prev90                         AS position_change,
        clk_prev90 / NULLIF(imp_prev90, 0)               AS ctr_prev90,
        imp_prev90_weekend / NULLIF(imp_prev90, 0)        AS weekend_share,
        imp_prev90_std / NULLIF(imp_prev90 / 90.0, 0)     AS impression_volatility
    FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,977 content items with enough history


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,clk_prev90,pos_last90,pos_prev90,imp_prev90_std,imp_prev90_weekend,position_change,ctr_prev90,weekend_share,impression_volatility
0,client_3ffa76342f366962,content_9b118b3509d3d54c,145.0,381.0,5.0,18.0,5.131176,4.048562,3.042989,125.0,1.082613,0.047244,0.328084,0.718816
1,client_3ffa76342f366962,content_cae1d5374958a649,1.0,213.0,0.0,0.0,0.000000,6.000236,2.424393,41.0,-6.000236,0.000000,0.192488,1.024391
2,client_3ffa76342f366962,content_dd66eecf9626cab8,2177.0,655.0,1.0,0.0,6.365859,6.374281,4.303194,193.0,-0.008422,0.000000,0.294656,0.591279
3,client_3ffa76342f366962,content_0674cc4ae0f68a90,5.0,107.0,1.0,1.0,1.000000,7.573871,2.188460,24.0,-6.573871,0.009346,0.224299,1.840760
4,client_3ffa76342f366962,content_0ffc84b3be2f3bc1,14.0,133.0,0.0,5.0,24.545455,9.532172,3.637680,26.0,15.013283,0.037594,0.195489,2.461588


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


joined: 120,977 rows


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,clk_prev90,pos_last90,pos_prev90,imp_prev90_std,imp_prev90_weekend,position_change,ctr_prev90,weekend_share,impression_volatility,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_3ffa76342f366962,content_9b118b3509d3d54c,145.0,381.0,5.0,18.0,5.131176,4.048562,3.042989,125.0,1.082613,0.047244,0.328084,0.718816,NaN,NaN,NaN,NaN,NaN,NaN
1,client_3ffa76342f366962,content_cae1d5374958a649,1.0,213.0,0.0,0.0,0.000000,6.000236,2.424393,41.0,-6.000236,0.000000,0.192488,1.024391,NaN,NaN,NaN,NaN,NaN,NaN
2,client_3ffa76342f366962,content_dd66eecf9626cab8,2177.0,655.0,1.0,0.0,6.365859,6.374281,4.303194,193.0,-0.008422,0.000000,0.294656,0.591279,2.0,0.00735,0.011943,1995.0,2135.0,0.934426
3,client_3ffa76342f366962,content_0674cc4ae0f68a90,5.0,107.0,1.0,1.0,1.000000,7.573871,2.188460,24.0,-6.573871,0.009346,0.224299,1.840760,NaN,NaN,NaN,NaN,NaN,NaN
4,client_3ffa76342f366962,content_0ffc84b3be2f3bc1,14.0,133.0,0.0,5.0,24.545455,9.532172,3.637680,26.0,15.013283,0.037594,0.195489,2.461588,NaN,NaN,NaN,NaN,NaN,NaN


## 5. A first honest model

Label: *did impressions decline by more than 20% over the 90-day window vs. the prior 90 days?*
— built only from columns that exist **before** the window we predict (prev-90 features +
query-mix predicting the last-90 outcome, so there's no leakage).

**Modified for Task 3:** instead of a random `train_test_split` (which can leak the *same client*
into both train and test, letting the model partly memorize client-specific baselines), this uses
**`GroupShuffleSplit` on `client_hash_id`** — whole clients go either to train or to test, never
both. This is a harder, more honest test of whether the model generalizes to a client it has
never seen.

The Random Forest is also tuned a bit from the original: `class_weight='balanced'` to correct for
the label imbalance, and `max_depth` / `min_samples_leaf` constraints to reduce overfitting on
noisy per-page signals — both aimed at genuine generalization rather than at inflating the
training-set score.

**Honest expectation:** a per-client split is *supposed* to be harder than a random split. If the
random-split number in the original notebook was partly inflated by client leakage, this number
can legitimately come out lower, even with better features and a tuned model — that's the whole
point of Task 3. Compare the two numbers and the classification report shape (precision/recall
per class), not just the headline accuracy.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last90'] < 0.8 * data['imp_prev90']).astype(int)

feature_cols = [
    'imp_prev90', 'pos_prev90', 'position_change', 'ctr_prev90',
    'weekend_share', 'impression_volatility',
    'visible_queries', 'rare_share', 'anon_share', 'top_query_share',
]
model_data = data.dropna(subset=feature_cols)
X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

print(f'train clients: {groups.iloc[train_idx].nunique()}   test clients: {groups.iloc[test_idx].nunique()}')
print(f'train rows: {len(X_tr):,}   test rows: {len(X_te):,}')

model = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
).fit(X_tr, y_tr)

print(f"\nbase rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}")
print(classification_report(y_te, model.predict(X_te), digits=3))

importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('feature importances:')
print(importances)

train clients: 30   test clients: 11
train rows: 64,165   test rows: 28,142

base rate (always predict majority): 0.523
              precision    recall  f1-score   support

           0      0.769     0.819     0.793     14720
           1      0.786     0.731     0.758     13422

    accuracy                          0.777     28142
   macro avg      0.778     0.775     0.775     28142
weighted avg      0.777     0.777     0.776     28142

feature importances:
imp_prev90               0.222095
position_change          0.162963
impression_volatility    0.150495
visible_queries          0.146591
rare_share               0.099929
pos_prev90               0.070909
ctr_prev90               0.052630
top_query_share          0.050682
anon_share               0.033916
weekend_share            0.009789
dtype: float64


### Reading the output

- **base rate** = what you'd get by always predicting the majority class. Only trust the model if
  it clearly beats this — not if it's within a point or two.
- **train clients / test clients** should not overlap — that's the whole fix from Task 3. If a
  client shows up in both counts something is wrong with the split.
- **feature importances** tells you which of the new Task-2 features (`position_change`,
  `ctr_prev90`, `weekend_share`, `impression_volatility`) actually carry signal vs. which ones are
  dead weight — drop the ones near zero rather than keeping them for their own sake.
- If precision/recall on the minority class is still weak, the next lever is usually **more
  history** (raise the `HAVING` threshold, or use a longer prior window) rather than more model
  tuning.

## Notes on what changed vs. the original notebook

```
I changed the feature window from 30 days to 90 days (Task 1), added position_change,
ctr_prev90, weekend_share, and impression_volatility as new engineered features (Task 2),
and replaced the random train_test_split with a GroupShuffleSplit on client_hash_id so
whole clients are held out (Task 3). I also switched the Random Forest to
class_weight='balanced' with constrained depth/leaf size to reduce overfitting.
```

Fill in your own numbers here once you've run it: content items before/after the window change,
and the base rate / precision / recall / f1 from the new classification report, so you can compare
directly against the original 30-day / random-split baseline.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.
